In [17]:
import sqlite3
from typing import Optional, List


class Passenger:
    def __init__(
        self,
        conn: sqlite3.Connection,
        passenger_id: Optional[int] = None,
        first_name: str = "",
        last_name: str = "",
        age: int = 0,
        sex: str = "",
        nationality: str = ""
    ):
        self.conn = conn
        self.passenger_id = passenger_id
        self.first_name = first_name
        self.last_name = last_name
        self.age = age
        self.sex = sex
        self.nationality = nationality

    @staticmethod
    def create_table(conn: sqlite3.Connection) -> None:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS passenger (
            passenger_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name TEXT NOT NULL,
            last_name TEXT NOT NULL,
            age INTEGER,
            sex TEXT,
            nationality TEXT
        );
        """)

    @classmethod
    def load_by_id(cls, conn: sqlite3.Connection, passenger_id: int) -> Optional["Passenger"]:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT passenger_id, first_name, last_name, age, sex, nationality
            FROM passenger
            WHERE passenger_id = ?
        """, (passenger_id,))
        row = cursor.fetchone()

        if row:
            return cls(conn, *row)
        return None

    def save(self) -> None:
        cursor = self.conn.cursor()

        if self.passenger_id is None:
            cursor.execute("""
                INSERT INTO passenger (first_name, last_name, age, sex, nationality)
                VALUES (?, ?, ?, ?, ?)
            """, (self.first_name, self.last_name, self.age, self.sex, self.nationality))
            self.passenger_id = cursor.lastrowid
        else:
            cursor.execute("""
                UPDATE passenger
                SET first_name = ?, last_name = ?, age = ?, sex = ?, nationality = ?
                WHERE passenger_id = ?
            """, (
                self.first_name,
                self.last_name,
                self.age,
                self.sex,
                self.nationality,
                self.passenger_id
            ))

        self.conn.commit()

    def delete(self) -> None:
        if self.passenger_id is not None:
            self.conn.execute(
                "DELETE FROM passenger WHERE passenger_id = ?",
                (self.passenger_id,)
            )
            self.conn.commit()

    def get_full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

    def get_assignments(self) -> List["CabinAssignment"]:
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT assignment_id, passenger_id, cabin_id, assigned_date, status
            FROM cabin_assignment
            WHERE passenger_id = ?
        """, (self.passenger_id,))
        rows = cursor.fetchall()

        return [
            CabinAssignment(self.conn, *row)
            for row in rows
        ]

    def assign_to_cabin(self, cabin: "Cabin", date: str, status: str = "active") -> None:
        assignment = CabinAssignment(
            self.conn,
            passenger_id=self.passenger_id,
            cabin_id=cabin.cabin_id,
            assigned_date=date,
            status=status
        )
        assignment.save()

    def unassign_from_cabin(self, cabin: "Cabin") -> None:
        self.conn.execute("""
            DELETE FROM cabin_assignment
            WHERE passenger_id = ? AND cabin_id = ?
        """, (self.passenger_id, cabin.cabin_id))
        self.conn.commit()


class Cabin:
    def __init__(
        self,
        conn: sqlite3.Connection,
        cabin_id: Optional[int] = None,
        cabin_class: str = "",
        number_of_beds: int = 1,
        room_price: float = 0.0,
        ship_level: int = 0
    ):
        self.conn = conn
        self.cabin_id = cabin_id
        self.cabin_class = cabin_class
        self.number_of_beds = number_of_beds
        self.room_price = room_price
        self.ship_level = ship_level

    @staticmethod
    def create_table(conn: sqlite3.Connection) -> None:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS cabin (
            cabin_id INTEGER PRIMARY KEY AUTOINCREMENT,
            cabin_class TEXT NOT NULL,
            number_of_beds INTEGER NOT NULL,
            room_price REAL NOT NULL,
            ship_level INTEGER
        );
        """)

    @classmethod
    def load_by_id(cls, conn: sqlite3.Connection, cabin_id: int) -> Optional["Cabin"]:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT cabin_id, cabin_class, number_of_beds, room_price, ship_level
            FROM cabin
            WHERE cabin_id = ?
        """, (cabin_id,))
        row = cursor.fetchone()

        if row:
            return cls(conn, *row)
        return None

    def save(self) -> None:
        cursor = self.conn.cursor()

        if self.cabin_id is None:
            cursor.execute("""
                INSERT INTO cabin (cabin_class, number_of_beds, room_price, ship_level)
                VALUES (?, ?, ?, ?)
            """, (
                self.cabin_class,
                self.number_of_beds,
                self.room_price,
                self.ship_level
            ))
            self.cabin_id = cursor.lastrowid
        else:
            cursor.execute("""
                UPDATE cabin
                SET cabin_class = ?, number_of_beds = ?, room_price = ?, ship_level = ?
                WHERE cabin_id = ?
            """, (
                self.cabin_class,
                self.number_of_beds,
                self.room_price,
                self.ship_level,
                self.cabin_id
            ))

        self.conn.commit()

    def delete(self) -> None:
        if self.cabin_id is not None:
            self.conn.execute(
                "DELETE FROM cabin WHERE cabin_id = ?",
                (self.cabin_id,)
            )
            self.conn.commit()

    def get_assignments(self) -> List["CabinAssignment"]:
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT assignment_id, passenger_id, cabin_id, assigned_date, status
            FROM cabin_assignment
            WHERE cabin_id = ?
        """, (self.cabin_id,))
        rows = cursor.fetchall()

        return [
            CabinAssignment(self.conn, *row)
            for row in rows
        ]

    def get_passengers(self) -> List[Passenger]:
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT p.passenger_id, p.first_name, p.last_name, p.age, p.sex, p.nationality
            FROM passenger p
            JOIN cabin_assignment ca ON p.passenger_id = ca.passenger_id
            WHERE ca.cabin_id = ?
        """, (self.cabin_id,))
        rows = cursor.fetchall()

        return [Passenger(self.conn, *row) for row in rows]

    def available_beds(self) -> int:
        occupied = len(self.get_passengers())
        return self.number_of_beds - occupied

    def is_full(self) -> bool:
        return self.available_beds() <= 0


class CabinAssignment:
    def __init__(
        self,
        conn: sqlite3.Connection,
        assignment_id: Optional[int] = None,
        passenger_id: Optional[int] = None,
        cabin_id: Optional[int] = None,
        assigned_date: str = "",
        status: str = "active"
    ):
        self.conn = conn
        self.assignment_id = assignment_id
        self.passenger_id = passenger_id
        self.cabin_id = cabin_id
        self.assigned_date = assigned_date
        self.status = status

    @staticmethod
    def create_table(conn: sqlite3.Connection) -> None:
        conn.execute("PRAGMA foreign_keys = ON;")
        conn.execute("""
        CREATE TABLE IF NOT EXISTS cabin_assignment (
            assignment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            passenger_id INTEGER NOT NULL,
            cabin_id INTEGER NOT NULL,
            assigned_date TEXT NOT NULL,
            status TEXT,
            FOREIGN KEY (passenger_id) REFERENCES passenger(passenger_id) ON DELETE CASCADE,
            FOREIGN KEY (cabin_id) REFERENCES cabin(cabin_id) ON DELETE CASCADE
        );
        """)

    @classmethod
    def load_by_id(cls, conn: sqlite3.Connection, assignment_id: int) -> Optional["CabinAssignment"]:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT assignment_id, passenger_id, cabin_id, assigned_date, status
            FROM cabin_assignment
            WHERE assignment_id = ?
        """, (assignment_id,))
        row = cursor.fetchone()

        if row:
            return cls(conn, *row)
        return None

    def save(self) -> None:
        cursor = self.conn.cursor()

        if self.assignment_id is None:
            cursor.execute("""
                INSERT INTO cabin_assignment (passenger_id, cabin_id, assigned_date, status)
                VALUES (?, ?, ?, ?)
            """, (
                self.passenger_id,
                self.cabin_id,
                self.assigned_date,
                self.status
            ))
            self.assignment_id = cursor.lastrowid
        else:
            cursor.execute("""
                UPDATE cabin_assignment
                SET passenger_id = ?, cabin_id = ?, assigned_date = ?, status = ?
                WHERE assignment_id = ?
            """, (
                self.passenger_id,
                self.cabin_id,
                self.assigned_date,
                self.status,
                self.assignment_id
            ))

        self.conn.commit()

    def delete(self) -> None:
        if self.assignment_id is not None:
            self.conn.execute("""
                DELETE FROM cabin_assignment
                WHERE assignment_id = ?
            """, (self.assignment_id,))
            self.conn.commit()

    def get_passenger(self) -> Optional[Passenger]:
        if self.passenger_id is None:
            return None
        return Passenger.load_by_id(self.conn, self.passenger_id)

    def get_cabin(self) -> Optional[Cabin]:
        if self.cabin_id is None:
            return None
        return Cabin.load_by_id(self.conn, self.cabin_id)


def create_database(db_name: str = "ship.db") -> sqlite3.Connection:
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON;")

    Passenger.create_table(conn)
    Cabin.create_table(conn)
    CabinAssignment.create_table(conn)

    return conn


def demo() -> None:
    conn = create_database()

    # Create passengers
    p1 = Passenger(conn, first_name="John", last_name="Smith", age=35, sex="M", nationality="British")
    p2 = Passenger(conn, first_name="Marie", last_name="Dubois", age=28, sex="F", nationality="French")
    p3 = Passenger(conn, first_name="Helen", last_name="Doe", age=102, sex="F", nationality="Italian")
    p1.save()
    p2.save()
    p3.save()

    # Create cabins
    c1 = Cabin(conn, cabin_class="First", number_of_beds=3, room_price=250.0, ship_level=1)
    c2 = Cabin(conn, cabin_class="Second", number_of_beds=4, room_price=120.0, ship_level=3)
    c3 = Cabin(conn, cabin_class="First", number_of_beds=1, room_price=250.0, ship_level=1)
    c1.save()
    c2.save()
    c3.save()

    # Assign passengers to a cabin
    p1.assign_to_cabin(c1, "1915-05-01")
    p2.assign_to_cabin(c1, "1915-05-01")
    p3.assign_to_cabin(c3, "1915-05-01")

    # Load records back using class methods
    loaded_passenger = Passenger.load_by_id(conn, p1.passenger_id)
    loaded_cabin = Cabin.load_by_id(conn, c1.cabin_id)
    
    print("Loaded passenger:", loaded_passenger.get_full_name() if loaded_passenger else "Not found")
    print("Loaded cabin:", loaded_cabin.cabin_class if loaded_cabin else "Not found")

    print(f"\nPassengers in cabin {c1.cabin_id}:")
    for passenger in c1.get_passengers():
        print("-", passenger.get_full_name(), "|", passenger.nationality)

    print("\nAvailable beds:", c1.available_beds())
    print("Is full:", c1.is_full())

    # printing exercise 1 results
    print(f"\nPassengers in cabin {c3.cabin_id}:")
    for passenger in c3.get_passengers():
        print("-", passenger.get_full_name(), "|", passenger.nationality)
        
    print("\nAvailable beds:", c3.available_beds())
    print("Is full:", c3.is_full())

    conn.close()


if __name__ == "__main__":
    demo()

Loaded passenger: John Smith
Loaded cabin: First

Passengers in cabin 20:
- John Smith | British
- Marie Dubois | French

Available beds: 1
Is full: False

Passengers in cabin 22:
- Helen Doe | Italian

Available beds: 0
Is full: True
